# Study 878 — Economic Policy Uncertainty ❓📰

**Does a spike in *policy uncertainty* tell you anything about the *future* — higher
volatility, or a return you get paid for bearing it?**

Baker, Bloom & Davis' newspaper-based **EPU** index is the canonical "how uncertain is
policy" gauge. It is sold on two stories: high EPU should precede **higher equity vol**
(the vol story) and **higher forward returns** as compensation (the risk-premium story).
We test both directly on the aggregate US market (1993-02-28 → 2026-06-30, 401 months).

> **Data-honesty note.** The real Baker-Bloom-Davis newspaper feed was **network-unreachable**
> from the build environment, so the signal here is a **labelled VIX proxy** (real CBOE
> implied vol) — a market-based stand-in, *never* the newspaper index. See `docs/references.md`.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `d922d69b4b52`); the live
cells run the fast synthetic control.*


## 1. The idea

When policy is uncertain — debt-ceiling brinkmanship, an election, a war — the newspapers fill with the word *uncertainty*. The story goes that markets get **choppier** (higher vol) and that you should be **paid extra** to hold stocks through the fog. The desk's prior is blunter: an uncertainty index is a *thermometer*, not a *crystal ball* — it spikes **with** the sell-off, not before the recovery.

In [1]:
import numpy as np, pandas as pd
R = dict(rv_t3=9.44, rv_r2_3=0.388, ret_t3=0.82, ret_r2_3=0.01,
         placebo_ret3_p=0.258, bh_sharpe=0.77, leanin_sharpe=0.49)
print('LEG 1  forward vol   on uncertainty (3m): HAC t = %+.2f  (R2 = %.3f)'
      % (R['rv_t3'], R['rv_r2_3']))
print('LEG 2  forward return on uncertainty (3m): HAC t = %+.2f  (R2 = %.3f)'
      % (R['ret_t3'], R['ret_r2_3']))
print('       return-leg block-shuffle placebo p = %.3f  (broken-link null)'
      % R['placebo_ret3_p'])
print('       timing on it: Sharpe %.2f  vs  buy-and-hold %.2f'
      % (R['leanin_sharpe'], R['bh_sharpe']))

LEG 1  forward vol   on uncertainty (3m): HAC t = +9.44  (R2 = 0.388)
LEG 2  forward return on uncertainty (3m): HAC t = +0.82  (R2 = 0.010)
       return-leg block-shuffle placebo p = 0.258  (broken-link null)
       timing on it: Sharpe 0.49  vs  buy-and-hold 0.77


## 2. Is the machinery even honest? A live synthetic control

We build a seeded toy world where the *previous* month's uncertainty genuinely drives *this* month's vol and return (`edge>0`), and check the detector recovers **both** legs — and stays silent on the null (`edge=0`). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from epu import data, strategy as st
null = st.synthetic_detect(*data.synthetic(360, 0.0, 0.0, 878), horizon=3)
planted = st.synthetic_detect(*data.synthetic(360, 0.02, 0.6, 878), horizon=3)
print('null    world: ret_t=%+.2f  rv_t=%+.2f  (should be ~0)' % (null['ret_t'], null['rv_t']))
print('planted world: ret_t=%+.2f  rv_t=%+.2f  (both light up)' % (planted['ret_t'], planted['rv_t']))

null    world: ret_t=+0.10  rv_t=+1.12  (should be ~0)
planted world: ret_t=+4.96  rv_t=+14.26  (both light up)


## 3. The honest verdict

On the real tape the **vol leg fires hard** (HAC *t* = **+9.44** at 3m) — but that is nearly **mechanical**: the VIX proxy *is* the market's implied vol, so of course it tracks realized vol; it's a *coincident thermometer*, not a forward edge. The **return leg — the part you'd actually get paid for — is dead**: HAC *t* = **+0.82** (R² ≈ 0.01), a placebo p of **0.26**, and what little post-2009 significance appears (*t* = +4.36) **vanishes pre-2009** (*t* = +0.09) — a single-era recovery-drift artefact. And no uncertainty-timed rule beats buy-and-hold (Sharpe 0.49 vs 0.77). **Signal: None** (uncertainty is contemporaneous), **Tradability: Mirage.**